In [16]:
import pandas as pd

df1 = pd.read_csv("Antworten_all.tsv", sep="\t")
df2 = pd.read_csv("Antworten_all_sandra.tsv", sep="\t")
df3 = pd.read_csv("evaluation_all - evaluation_all.csv (2).tsv", sep="\t")

df = df1.merge(df2, on=["id", "question_de", "answer_de", "max_points", "keywords", "question_type",], suffixes=("_alex", "_sandra"))
df = df.merge(df3, on=["id", "question_de", "answer_de", "max_points", "keywords", "question_type"], suffixes=("", "_maurice"))

In [17]:
len(df)

10

In [18]:
df.columns

Index(['id', 'question_de', 'answer_de', 'student_answer_alex',
       'Fehlerbild_alex', 'max_points', 'keywords', 'question_type',
       'transkript_stt_model_alex', 'delta_original_transkript',
       'error_type_alex', 'human_score_alex', 'human_feedback_bullets',
       'human_feedback_alex', 'rueckfrage_alex', 'Unnamed: 15_alex',
       'Unnamed: 16_alex', 'Unnamed: 17_alex', 'Unnamed: 18_alex',
       'Unnamed: 19_alex', 'Unnamed: 20_alex', 'student_answer_sandra',
       'transkript_stt_model_sandra', 'delta_original_answer',
       'error_type_sandra', 'Fehlerbild_sandra', 'human_feedback_sandra',
       'rueckfrage_sandra', 'human_score_sandra', 'Unnamed: 14',
       'Unnamed: 15_sandra', 'Unnamed: 16_sandra', 'Unnamed: 17_sandra',
       'Unnamed: 18_sandra', 'Unnamed: 19_sandra', 'Unnamed: 20_sandra',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 0', 'question_en', 'answer_en',
       'translation_similari

In [19]:
df = df[['id', 'question_de', 'answer_de', 'student_answer_alex',
       'max_points', 'keywords', 'question_type',
       'transkript_stt_model_alex',
       'error_type_alex', 'human_score_alex', 'human_feedback_bullets',
       'human_feedback_alex', 'rueckfrage_alex', 'student_answer_sandra',
       'transkript_stt_model_sandra',
       'error_type_sandra', 'human_feedback_sandra',
       'rueckfrage_sandra', 'human_score_sandra', 'transkript_stt_model', 'error_type', 'human_score',
       'human_feedback_bullets_maurice', 'human_feedback', 'rueckfrage']]

In [20]:
# --- Semantic agreement for human_feedback (3 raters) ---
# Computes pairwise cosine similarities + an interval [lower, upper] per row.

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1) Pick your 3 human feedback columns (adjust if needed)
RATER_COLS = ["human_feedback_alex", "human_feedback_sandra", "human_feedback"]  # <- check 3rd one is Maurice
# If Maurice is in another column, e.g. "human_feedback_bullets_maurice", put that here instead.

# 2) Basic cleaning (optional but helps)
def clean_text(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x)
    x = " ".join(x.split())  # collapse whitespace/newlines
    return x

for c in RATER_COLS:
    df[c] = df[c].apply(clean_text)

# 3) Load an embedding model (good multilingual default)
# You can also try: "intfloat/multilingual-e5-base" (then add "query: " prefixes), but this is plug&play.
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 4) Efficient embedding with caching (speeds up if texts repeat)
def embed_texts_unique(texts, batch_size=64):
    unique = list(dict.fromkeys(texts))  # preserves order, removes duplicates
    embs = model.encode(unique, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    emb_map = {t: e for t, e in zip(unique, embs)}
    return np.vstack([emb_map[t] for t in texts])

# Embed each rater column
E = {}
for c in RATER_COLS:
    E[c] = embed_texts_unique(df[c].tolist(), batch_size=64)

# 5) Pairwise cosine similarity (since embeddings are normalized: cosine = dot product)
a, b, c = RATER_COLS
sim_ab = np.sum(E[a] * E[b], axis=1)
sim_ac = np.sum(E[a] * E[c], axis=1)
sim_bc = np.sum(E[b] * E[c], axis=1)

# 6) Per-row interval + summary stats
sims = np.vstack([sim_ab, sim_ac, sim_bc]).T  # shape (n_rows, 3)

df["sim_ab"] = sim_ab
df["sim_ac"] = sim_ac
df["sim_bc"] = sim_bc

df["sem_sim_lower"] = sims.min(axis=1)   # lower bound of human semantic agreement
df["sem_sim_upper"] = sims.max(axis=1)   # upper bound
df["sem_sim_range"] = df["sem_sim_upper"] - df["sem_sim_lower"]
df["sem_sim_mean_pairwise"] = sims.mean(axis=1)
df["sem_sim_median_pairwise"] = np.median(sims, axis=1)

# 7) Quick aggregated view (overall)
summary = df[["sem_sim_lower","sem_sim_upper","sem_sim_range","sem_sim_mean_pairwise"]].describe()
print(summary)

# 8) Optional: per-question_type / per-error_type breakdowns (useful to find “hard” items)
# print(df.groupby("question_type")["sem_sim_range"].describe())
# print(df.groupby("error_type")["sem_sim_range"].describe())

# 9) Optional: flag rows with low agreement (e.g. lower bound < 0.6)
df["low_semantic_agreement"] = df["sem_sim_lower"] < 0.60
print("Low-agreement share:", df["low_semantic_agreement"].mean())

/Users/aleks/Desktop/FernUni_Hagen/EchoLearnBrainstorming/.conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]

       sem_sim_lower  sem_sim_upper  sem_sim_range  sem_sim_mean_pairwise
count      10.000000      10.000000      10.000000              10.000000
mean        0.582190       0.758353       0.176163               0.679926
std         0.133490       0.069303       0.138278               0.078986
min         0.331036       0.618355       0.038296               0.576160
25%         0.536224       0.725590       0.077516               0.618534
50%         0.592074       0.767701       0.156847               0.675196
75%         0.645173       0.814256       0.206635               0.724443
max         0.792479       0.830775       0.498915               0.810428
Low-agreement share: 0.5


In [39]:
import numpy as np

score_cols = [
    "human_score_alex",
    "human_score_sandra",
    "human_score"   # Maurice
]

df["human_score_alex"] = df["human_score_alex"].str.replace(",", ".").astype(float)
df["human_score_sandra"] = df["human_score_sandra"].str.replace(",", ".").astype(float)
df["human_score"] = df["human_score"].str.replace(",", ".").astype(float)

scores = df[score_cols].apply(pd.to_numeric, errors='coerce').values.astype(float)
df["human_score_median"] = np.median(scores, axis=1)
for col in score_cols:
    df[f"dev_pct_{col}"] = (
        (df[col] - df["human_score_median"]).abs()
        / df["max_points"]
    )
max_deviation = (
    df[[f"dev_pct_{c}" for c in score_cols]]
    .max()
    .sort_values(ascending=False)
)

print(max_deviation)

dev_pct_human_score           0.375
dev_pct_human_score_sandra    0.250
dev_pct_human_score_alex      0.200
dtype: float64


In [40]:
error_cols = [
    "error_type_alex",
    "error_type_sandra",
    "error_type"  # Maurice
]
errors = df[error_cols].values

df["error_lower"] = errors.min(axis=1)
df["error_upper"] = errors.max(axis=1)
df["error_range"] = df["error_upper"] - df["error_lower"]
MAX_ERROR = 3

df["error_range_pct"] = df["error_range"] / MAX_ERROR
df["error_median"] = np.median(errors, axis=1)

for col in error_cols:
    df[f"dev_error_{col}"] = (df[col] - df["error_median"]).abs()
    df[f"dev_error_pct_{col}"] = df[f"dev_error_{col}"] / MAX_ERROR
max_dev_error = (
    df[[f"dev_error_pct_{c}" for c in error_cols]]
    .max()
    .sort_values(ascending=False)
)

print(max_dev_error)
df["error_range_pct"].describe()

dev_error_pct_error_type_sandra    1.000000
dev_error_pct_error_type           0.666667
dev_error_pct_error_type_alex      0.333333
dtype: float64


count    10.000000
mean      0.433333
std       0.386501
min       0.000000
25%       0.083333
50%       0.333333
75%       0.666667
max       1.000000
Name: error_range_pct, dtype: float64

In [41]:
rq_cols = [
    "rueckfrage_alex",
    "rueckfrage_sandra",
    "rueckfrage"   # Maurice
]
rq = df[rq_cols].values

df["rq_lower"] = rq.min(axis=1)
df["rq_upper"] = rq.max(axis=1)
df["rq_range"] = df["rq_upper"] - df["rq_lower"]
MAX_RQ = 3
df["rq_range_pct"] = df["rq_range"] / MAX_RQ
for col in rq_cols:
    df[f"rq_binary_{col}"] = (df[col] > 0).astype(int)
rq_bin = df[[f"rq_binary_{c}" for c in rq_cols]].values

df["rq_bin_lower"] = rq_bin.min(axis=1)
df["rq_bin_upper"] = rq_bin.max(axis=1)
df["rq_bin_disagree"] = df["rq_bin_lower"] != df["rq_bin_upper"]
df["rq_median"] = np.median(rq, axis=1)

for col in rq_cols:
    df[f"dev_rq_{col}"] = (df[col] - df["rq_median"]).abs()
    df[f"dev_rq_pct_{col}"] = df[f"dev_rq_{col}"] / MAX_RQ
max_dev_rq = (
    df[[f"dev_rq_pct_{c}" for c in rq_cols]]
    .max()
    .sort_values(ascending=False)
)
print(max_dev_rq)
df["rq_bin_disagree"].mean()

dev_rq_pct_rueckfrage           0.666667
dev_rq_pct_rueckfrage_alex      0.333333
dev_rq_pct_rueckfrage_sandra    0.000000
dtype: float64


0.2